# 05 · Ablation over the state vector  *(branch `exp/05-rl-ablation`)* — the core result

Brief §3.4 (a). **Same agent (PPO), same hyperparameters**, four states, 5 seeds each:

| | state | dim | answers |
|---|---|---|---|
| A | `[position, time-in-position] + 8 factors` | 10 | control |
| B | A + sentiment | 11 | RQ1 |
| C | A + risk | 11 | isolates risk |
| D | A + both | 12 | RQ3 (vs B) |

LLM signals are the **previous completed day's** aggregate (causal). `LEAK_CHECK` additionally runs D with the
dataset's same-day alignment — the difference measures how much the look-ahead would have inflated the result.

**Tuned re-run** (`TUNED = True`, after 06): the first run used PPO with default hyperparameters, which ends up holding 0-1 static positions in every configuration, so the signals cannot show an effect. The re-run repeats A-D with the agent that has the best mean validation Sharpe in 06, with its tuned hyperparameters (the same for all four states) and the 5-validation-trade eligibility rule, as a separate experiment. Its state D is that agent's 06 run (identical configuration and seeds; runs are deterministic), so only A-C are trained here.


In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
TUNED = True       # True: re-run A-D with the agent tuned in 06 (see the configuration cell); False: first run (PPO)
BRANCH = "exp/06-agent-comparison" if TUNED else "exp/05-rl-ablation"   # agents other than PPO are added in 06
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "05_rl_ablation")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import results as R
SEEDS = [0, 1, 2, 3, 4]
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
LEAK_CHECK = True
F = paths.factor_dir(L)

# TUNED = False: the first run - notebook PPO defaults, experiment "05_ablation" (kept as it is).
# TUNED = True:  the agent with the best mean validation Sharpe in 06, with its tuned hyperparameters
#                (identical for A-D) and the same eligibility rule; experiment "05_ablation_<agent>_mt5".
#                Needs 06 to have finished.
MIN_TRADES = 5     # TUNED is set in the setup cell, because it also decides which code branch is checked out
if TUNED:
    res = R.load_results(L.runs)
    a6 = res[(res.experiment == f"06_agents_mt{MIN_TRADES}") & ~res.agent.isin(["buy_and_hold", "cash", "random"])]
    AGENT = a6.groupby("agent").val_sharpe.mean().idxmax()
    hp = json.loads((L.runs / "tuning" / f"best_hp_mt{MIN_TRADES}.json").read_text())[AGENT]
    AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
    EXP, NAME = f"05_ablation_{AGENT}_mt{MIN_TRADES}", f"ablation_{AGENT}_mt{MIN_TRADES}"
    from rltrade.agents import AGENTS as AGENT_CLASSES
    assert AGENT in AGENT_CLASSES, f"agent {AGENT!r} is not in this code version - check BRANCH in the setup cell"
else:
    AGENT, AGENT_HP, ENV_OVER = "ppo", {}, {}          # notebook PPO defaults - identical for A-D
    EXP, NAME = "05_ablation", "ablation"
# With the tuned agent, state D is exactly that agent's 06 run (same configuration, seeds and code, and runs
# are deterministic - checked in 07, whose net_asset_change runs reproduce 06 exactly), so it is reused.
REUSE_06_FOR_D = TUNED
STATES = ("A", "B", "C") if REUSE_06_FOR_D else ("A", "B", "C", "D")
print("agent:", AGENT, AGENT_HP, ENV_OVER, "| experiment:", EXP, "| states run here:", STATES)

In [ ]:
for state in STATES:
    run_seeds(RunSpec(EXP, AGENT, env=EnvConfig(state=state, **ENV_OVER), agent_hp=AGENT_HP, min_val_trades=MIN_TRADES,
                      total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS, L.prepared, F, L.runs)
run_seeds(RunSpec(EXP, "buy_and_hold", env=EnvConfig(state="D")), [0], L.prepared, F, L.runs)
if TUNED:
    run_seeds(RunSpec(EXP, "cash", env=EnvConfig(state="D")), [0], L.prepared, F, L.runs)      # "do nothing" reference
if LEAK_CHECK:
    run_seeds(RunSpec(EXP, AGENT, env=EnvConfig(state="D", signal_source="sameday", **ENV_OVER), agent_hp=AGENT_HP,
                      min_val_trades=MIN_TRADES, total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY),
              SEEDS, L.prepared, F, L.runs)

## Results

In [ ]:
res = R.load_results(L.runs); df = res[res.experiment == EXP]
if REUSE_06_FOR_D:     # state D = this agent's 06 runs (see the configuration cell)
    df = pd.concat([df, res[(res.experiment == f"06_agents_mt{MIN_TRADES}") & (res.agent == AGENT)]], ignore_index=True)
ab = df[(df.agent == AGENT) & (df.signal_source == "lagged")]
sm = R.summarise(ab)
table = R.markdown_table(sm, group=("state",))
print(table)
if "selection_eligible" in ab:     # seeds where no checkpoint met the eligibility rule (fallback checkpoint)
    print("seeds without an eligible checkpoint:",
          ab[ab.selection_eligible.astype(str) == "False"].groupby("state").size().to_dict() or "none")
R.plot_ablation(sm, L.report / f"{NAME}_bars.png",
                title=f"Ablation over the state vector, {AGENT.upper()} (test, mean ± std over seeds)"); plt.show()

def compare(a, b, metric="test_sharpe"):
    x, y = ab[ab.state == a][metric].values, ab[ab.state == b][metric].values
    t = stats.ttest_ind(x, y, equal_var=False)
    return {"pair": f"{a} - {b}", "diff_mean": x.mean() - y.mean(), "welch_t": t.statistic, "p": t.pvalue, "n": (len(x), len(y))}
tests = pd.DataFrame([compare(*p) for p in (("B", "A"), ("C", "A"), ("D", "A"), ("D", "B"))])
print(tests.round(4))       # n = 5 per group: treat p-values as indicative only
(L.report / f"{NAME}.md").write_text(f"# Ablation A-D, {AGENT}"
                                     + (" tuned in 06, >= 5 validation trades" if TUNED else " default hyperparameters")
                                     + " (test, net of costs)\n\n" + table + "\n\n" + R.df_markdown(tests, 4) + "\n", encoding="utf-8")

In [ ]:
groups = {f"{AGENT.upper()} {s}": ab[ab.state == s] for s in "ABCD"}
groups["Buy & hold"] = df[df.agent == "buy_and_hold"]
if (df.agent == "cash").any():
    groups["Cash"] = df[df.agent == "cash"]
R.plot_curves(df, L.runs, groups, path=L.report / f"{NAME}_equity.png"); plt.show()
R.plot_curves(df, L.runs, groups, kind="drawdown", path=L.report / f"{NAME}_drawdown.png"); plt.show()
if LEAK_CHECK:
    lk = df[(df.agent == AGENT) & (df.state == "D")]
    print(R.markdown_table(R.summarise(lk), group=("state", "signal_source")))